<a href="https://colab.research.google.com/github/Danielbarz/analisis-sentimen-padel/blob/main/UTS_Pre-Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install & Import Libraries

In [1]:
!pip install openpyxl pandas nltk PySastrawi scikit-learn -q
import nltk
nltk.download('punkt')
nltk.download('stopwords')

import pandas as pd
import numpy as np
import re
import os
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print('Libraries ready.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.6/210.6 kB 3.5 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Libraries ready.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## 2. Load Data dari File Excel Hasil Scraping

In [4]:
from google.colab import files
import io

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(io.BytesIO(uploaded[filename]), engine='openpyxl')
print(f'Data loaded: {df.shape[0]} baris x {df.shape[1]} kolom')
print(f'Kolom: {list(df.columns)}')
df.head(3)

Saving Padel_AI_Hybrid.xlsx to Padel_AI_Hybrid.xlsx
Data loaded: 789 baris x 10 kolom
Kolom: ['judul', 'url', 'sumber', 'tanggal', 'keyword', 'metode_scraping', 'konten', 'konten_bersih', 'teks_analisis', 'label']


,judul,url,sumber,tanggal,keyword,metode_scraping,konten,konten_bersih,teks_analisis,label
0,"Riset: Laju Properti 2026 Moderat, Lapangan Pa...",https://news.google.com/rss/articles/CBMixgFBV...,CNN Indonesia,2026-01-26 08:00:00,padel trend Indonesia,google_news_180d,"Riset: Laju Properti 2026 Moderat, Lapangan Pa...",riset laju properti 2026 moderat lapangan pade...,riset laju properti 2026 moderat lapangan pade...,1
1,Berapa Biaya Bangun Lapangan Padel? Ini Rincia...,https://news.google.com/rss/articles/CBMiowFBV...,Bloomberg Technoz,2026-02-28 08:00:00,padel,google_news_180d,Berapa Biaya Bangun Lapangan Padel? Ini Rincia...,berapa biaya bangun lapangan padel ini rincian...,berapa biaya bangun lapangan padel ini rincian...,1
2,"Maple Padel, Lapangan Padel Rooftop Pertama di...",https://news.google.com/rss/articles/CBMiYkFVX...,RRI.co.id,2026-02-06 08:00:00,komunitas padel,google_news_180d,"Maple Padel, Lapangan Padel Rooftop Pertama di...",maple padel lapangan padel rooftop pertama di ...,maple padel lapangan padel rooftop pertama di ...,1


## 3. Inspeksi Awal & Statistik Data

In [5]:
print(f'Total artikel      : {len(df)}')
print(f'Kolom kosong:\n{df.isnull().sum().to_string()}')
print(f'\nDistribusi label:\n{df["label"].value_counts().to_string()}')
print(f'\nKonten kosong      : {(df["konten"].isna() | (df["konten"] == "")).sum()}')
print(f'Teks analisis kosong: {(df["teks_analisis"].isna() | (df["teks_analisis"] == "")).sum()}')

Total artikel      : 789
Kolom kosong:
judul                0
url                  0
sumber               0
tanggal            138
keyword              0
metode_scraping      0
konten               8
konten_bersih        8
teks_analisis        0
label                0

Distribusi label:
label
1    711
2     70
0      8

Konten kosong      : 8
Teks analisis kosong: 0


## 4. Pembersihan HTML, URL, dan Whitespace

In [6]:
def clean_basic(text):
    if not text or pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'<[^>]+>', '', text)            # hapus tag HTML
    text = re.sub(r'&[a-z]+;', ' ', text)          # hapus HTML entities: &nbsp; &amp; dll
    text = re.sub(r'http\S+|www\.\S+', '', text)   # hapus URL
    text = re.sub(r'\s+', ' ', text)               # normalize whitespace
    return text.strip()

df['konten_clean'] = df['konten'].apply(clean_basic)
df['judul_clean'] = df['judul'].apply(clean_basic)

print('HTML, URL, whitespace cleaned.')
print(df[['judul', 'judul_clean']].head(3).to_string())

HTML, URL, whitespace cleaned.
                                                                                judul                                                                         judul_clean
0  Riset: Laju Properti 2026 Moderat, Lapangan Padel Justru Meningkat - CNN Indonesia  Riset: Laju Properti 2026 Moderat, Lapangan Padel Justru Meningkat - CNN Indonesia
1              Berapa Biaya Bangun Lapangan Padel? Ini Rinciannya - Bloomberg Technoz              Berapa Biaya Bangun Lapangan Padel? Ini Rinciannya - Bloomberg Technoz
2                   Maple Padel, Lapangan Padel Rooftop Pertama di Jateng - RRI.co.id                   Maple Padel, Lapangan Padel Rooftop Pertama di Jateng - RRI.co.id


## 5. Normalisasi Teks (Lowercase, Strip, Hapus Non-Alfanumerik)

In [7]:
def normalize(text):
    if not text:
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)   # hapus non-alfanumerik
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['konten_norm'] = df['konten_clean'].apply(normalize)
df['judul_norm'] = df['judul_clean'].apply(normalize)

print('Normalisasi selesai.')
print(df[['judul_clean', 'judul_norm']].head(3).to_string())

Normalisasi selesai.
                                                                          judul_clean                                                                      judul_norm
0  Riset: Laju Properti 2026 Moderat, Lapangan Padel Justru Meningkat - CNN Indonesia  riset laju properti 2026 moderat lapangan padel justru meningkat cnn indonesia
1              Berapa Biaya Bangun Lapangan Padel? Ini Rinciannya - Bloomberg Technoz             berapa biaya bangun lapangan padel ini rinciannya bloomberg technoz
2                   Maple Padel, Lapangan Padel Rooftop Pertama di Jateng - RRI.co.id                    maple padel lapangan padel rooftop pertama di jateng rricoid


## 6. Drop Artikel Tanpa Konten

In [10]:
before = len(df)
df = df[df['konten_norm'] != ''].reset_index(drop=True)
print(f'Dropped : {before - len(df)} artikel tanpa konten')
print(f'Sisa    : {len(df)} artikel')

Dropped : 0 artikel tanpa konten
Sisa    : 789 artikel


## 7. Tokenisasi Teks (word_tokenize per token)

In [11]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [12]:
def tokenize(text):
    if not text:
        return []
    return word_tokenize(text)   # hapus language='indonesian'

df['tokens'] = df['konten_norm'].apply(tokenize)

print('Tokenisasi selesai.')
print(f'Rata-rata token per artikel: {df["tokens"].apply(len).mean():.0f}')
print(f'Contoh token:\n{df["tokens"].iloc[0][:20]}')

Tokenisasi selesai.
Rata-rata token per artikel: 70
Contoh token:
['riset', 'laju', 'properti', '2026', 'moderat', 'lapangan', 'padel', 'justru', 'meningkat', 'cnn', 'indonesia']


## 8. Stopword Removal (Bahasa Indonesia + Custom List)


cek kata frekuensi tinggi yang tidak bermakna

In [13]:
from collections import Counter

all_tokens = [t for tokens in df['tokens'] for t in tokens]
freq = Counter(all_tokens).most_common(50)

import pandas as pd
pd.DataFrame(freq, columns=['kata', 'frekuensi'])

,kata,frekuensi
0,padel,1953
1,yang,1415
2,di,1331
3,dan,1159
4,lapangan,823
5,ini,639
6,untuk,585
7,dengan,568
8,olahraga,537
9,dari,432


In [14]:
base_stopwords = set(stopwords.words('indonesian'))

custom_stopwords = {
    # noise HTML & portal
    'nbsp', 'scroll', 'continue', 'content', 'advertisement',
    'detikers', 'kompascom', 'detikcom', 'rricoid', 'sindonews',
    'tvonenews', 'gambas', 'video', 'detik', 'artikel', 'baca',
    # kata umum tidak bermakna untuk sentimen (dari hasil freq check)
    'yang', 'di', 'dan', 'ini', 'untuk', 'dengan', 'dari', 'juga',
    'itu', 'dalam', 'lebih', 'ada', 'bisa', 'tidak', 'menjadi',
    'hingga', 'pada', 'jadi', 'akan', 'bagi', 'saat', 'karena',
    'sudah', 'mulai', 'sebagai', 'tersebut', 'atau', 'ke', 'satu',
    'dapat', 'hanya', 'tak', 'tetap', 'seperti',
}

all_stopwords = base_stopwords.union(custom_stopwords)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in all_stopwords and len(t) > 2]

df['tokens_clean'] = df['tokens'].apply(remove_stopwords)

print('Stopword removal selesai.')
print(f'Rata-rata token setelah removal: {df["tokens_clean"].apply(len).mean():.0f}')

Stopword removal selesai.
Rata-rata token setelah removal: 42


## 9. Stemming (PySastrawi - Normalisasi Kata Bahasa Indonesia)

In [20]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens):
    return [stemmer.stem(t) for t in tokens]

df['tokens_stemmed'] = df['tokens_clean'].apply(stem_tokens)
df['konten_stemmed'] = df['tokens_stemmed'].apply(lambda t: ' '.join(t))

df[['konten_norm', 'tokens_clean', 'tokens_stemmed', 'konten_stemmed']].head(3)

,konten_norm,tokens_clean,tokens_stemmed,konten_stemmed
0,riset laju properti 2026 moderat lapangan pade...,"[riset, laju, properti, 2026, moderat, lapanga...","[riset, laju, properti, 2026, moderat, lapang,...",riset laju properti 2026 moderat lapang padel ...
1,berapa biaya bangun lapangan padel ini rincian...,"[biaya, bangun, lapangan, padel, rinciannya, b...","[biaya, bangun, lapang, padel, rinciannya, blo...",biaya bangun lapang padel rinciannya bloomberg...
2,maple padel lapangan padel rooftop pertama di ...,"[maple, padel, lapangan, padel, rooftop, jateng]","[maple, padel, lapang, padel, rooftop, jateng]",maple padel lapang padel rooftop jateng


## 10. POS Tagging (NLTK - tag: NN, VB, JJ, dll)

In [21]:
import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

def pos_tag_tokens(tokens):
    return nltk.pos_tag(tokens)

df['pos_tags'] = df['tokens_clean'].apply(pos_tag_tokens)

df[['judul', 'tokens_clean', 'pos_tags']].head(3)

,judul,tokens_clean,pos_tags
0,"Riset: Laju Properti 2026 Moderat, Lapangan Pa...","[riset, laju, properti, 2026, moderat, lapanga...","[(riset, NN), (laju, NN), (properti, NN), (202..."
1,Berapa Biaya Bangun Lapangan Padel? Ini Rincia...,"[biaya, bangun, lapangan, padel, rinciannya, b...","[(biaya, NN), (bangun, NN), (lapangan, JJ), (p..."
2,"Maple Padel, Lapangan Padel Rooftop Pertama di...","[maple, padel, lapangan, padel, rooftop, jateng]","[(maple, JJ), (padel, NN), (lapangan, JJ), (pa..."


## 11. TF-IDF Vectorization (TfidfVectorizer sklearn - fitur analisis dan baseline)

In [22]:
tfidf = TfidfVectorizer(max_features=5000, min_df=2, max_df=0.95)
tfidf_matrix = tfidf.fit_transform(df['konten_stemmed'])

df['tfidf_vector'] = list(tfidf_matrix.toarray())

feature_names = tfidf.get_feature_names_out()
tfidf_preview = pd.DataFrame(
    tfidf_matrix.toarray()[:5],
    columns=feature_names
)
tfidf_preview.iloc[:, :10]

,0600,088,0900,092,100,1000,105,10x20,110,1112
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [24]:
top_words = []
for i in range(5):
    row = tfidf_matrix.getrow(i)
    top_idx = row.toarray()[0].argsort()[::-1][:5]
    words = [(feature_names[j], round(tfidf_matrix[i, j], 4)) for j in top_idx if tfidf_matrix[i, j] > 0]
    top_words.append({'artikel': df['judul'].iloc[i][:40], 'top_kata': words})

pd.DataFrame(top_words)

,artikel,top_kata
0,"Riset: Laju Properti 2026 Moderat, Lapan","[(riset, 0.4749), (laju, 0.4541), (properti, 0..."
1,Berapa Biaya Bangun Lapangan Padel? Ini,"[(rinciannya, 0.554), (bloomberg, 0.4614), (te..."
2,"Maple Padel, Lapangan Padel Rooftop Pert","[(rooftop, 0.7774), (jateng, 0.5985), (lapang,..."
3,Pramono Resmi Larang Pembangunan Lapanga,"[(zona, 0.4479), (cnn, 0.4385), (larang, 0.397..."
4,Baru Pertama Main Padel? Ini Cara Bermai,"[(main, 0.5202), (ajar, 0.2995), (dasar, 0.294..."


## 12. Pemetaan Label & Split Data (Train 80% / Val 10% / Test 10%)
Label: 0 = Negatif, 1 = Netral, 2 = Positif

In [18]:
df['label'] = pd.to_numeric(df['label'], errors='coerce').fillna(1).astype(int)
df = df[df['label'].isin([0, 1, 2])].reset_index(drop=True)

label_map = {0: 'negatif', 1: 'netral', 2: 'positif'}
df['label_text'] = df['label'].map(label_map)

X = df['konten_stemmed']
y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}')

df[['judul', 'label', 'label_text']].head(5)

Train : 631 (80%)
Val   : 79 (10%)
Test  : 79 (10%)

Distribusi label train:
label
1    569
2     56
0      6


## 13. Output DataFrame Final
Kolom: konten_bersih, pos_tags, tfidf_vector, label

In [23]:
cols_output = [
    'judul', 'url', 'sumber', 'tanggal', 'keyword', 'metode_scraping',
    'konten', 'konten_norm', 'konten_stemmed', 'tokens_clean',
    'pos_tags', 'tfidf_vector', 'label', 'label_text'
]

df_final = df[cols_output].copy()

ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
os.makedirs('data', exist_ok=True)

df_final.drop(columns=['tfidf_vector', 'pos_tags', 'tokens_clean']).to_csv(
    f'data/padel_preprocessed_{ts}.csv', index=False, encoding='utf-8-sig'
)
df_final.drop(columns=['tfidf_vector', 'pos_tags', 'tokens_clean']).to_excel(
    f'data/padel_preprocessed_{ts}.xlsx', index=False
)

print(f'Saved  : data/padel_preprocessed_{ts}.csv')
print(f'Total  : {len(df_final)} artikel')

df_final[['judul', 'konten_stemmed', 'label', 'label_text']].head(5)

Saved  : data/padel_preprocessed_20260504_133906.csv
Total  : 789 artikel


,judul,konten_stemmed,label,label_text
0,"Riset: Laju Properti 2026 Moderat, Lapangan Pa...",riset laju properti 2026 moderat lapang padel ...,1,netral
1,Berapa Biaya Bangun Lapangan Padel? Ini Rincia...,biaya bangun lapang padel rinciannya bloomberg...,1,netral
2,"Maple Padel, Lapangan Padel Rooftop Pertama di...",maple padel lapang padel rooftop jateng,1,netral
3,Pramono Resmi Larang Pembangunan Lapangan Pade...,pramono resmi larang bangun lapang padel zona ...,1,netral
4,Baru Pertama Main Padel? Ini Cara Bermain yang...,padel olahraga viral gari indonesia cinta olah...,2,positif
